In [ ]:
import lightning as L
import torch
from microvit.models.attention.vanilla import VanillaAttention
from microvit.models.patch_embed import ConvPatchEmbed
from microvit.models.transformer.vanilla import VanillaTransformerBlock
from microvit.models.vit import ViT
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

In [ ]:
embed_dim = 64
num_heads = 2
depth = 4
img_size = 28
patch_size = 4
in_channels = 1

patch_embed = ConvPatchEmbed(
    img_size=img_size,
    patch_size=patch_size,
    in_channels=in_channels,
    embed_dim=embed_dim,
)


def block_factory(drop_path):
    return VanillaTransformerBlock(
        attention=VanillaAttention(
            embed_dim=embed_dim, num_heads=num_heads, dropout=0.0
        ),
        drop_path=drop_path,
    )


vit = ViT(
    embed_dim=embed_dim,
    patch_embed=patch_embed,
    block_factory=block_factory,
    depth=depth,
    num_classes=10,
)

In [ ]:
class MNISTModule(L.LightningModule):
    def __init__(self, model: nn.Module):
        super().__init__()
        self.model = model
        self.loss = nn.CrossEntropyLoss()

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self.model(x)
        loss = self.loss(logits, y)
        acc = (logits.argmax(dim=-1) == y).float().mean()
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self.model(x)
        loss = self.loss(logits, y)
        acc = (logits.argmax(dim=-1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

In [ ]:
transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),  # [-1, 1]
    ]
)

full_train = datasets.MNIST(
    root="data/", train=True, download=True, transform=transform
)
test_set = datasets.MNIST(root="data/", train=False, download=True, transform=transform)

train_set, val_set = random_split(full_train, [55000, 5000])

train_loader = DataLoader(train_set, batch_size=256, shuffle=True, num_workers=0)
val_loader = DataLoader(val_set, batch_size=256, num_workers=0)

In [ ]:
module = MNISTModule(vit)

trainer = L.Trainer(
    max_epochs=50,
    accelerator="mps",
    log_every_n_steps=10,
)

trainer.fit(module, train_loader, val_loader)